# Module 5: Graph Algorithms

**Prerequisites:** Modules 1–4 (Introduction, Logic & Proofs, Recursion & Induction, Data Structures)

Graphs are one of the most versatile data structures in computer science. In this module, we represent graphs in ACL2, implement classic graph algorithms (DFS, BFS, Dijkstra), and *prove* their correctness properties.

## 1. Graph Representation

We represent graphs as *adjacency lists* — an alist mapping each node to its list of neighbors.

For an **unweighted** graph:
```
((A . (B C))     ; A connects to B and C
 (B . (A D))     ; B connects to A and D
 ...)
```

For a **weighted** graph (used in Dijkstra's algorithm):
```
((A . ((B . 4) (C . 2)))    ; A→B cost 4, A→C cost 2
 (B . ((D . 3)))             ; B→D cost 3
 ...)
```

In [ ]:
; Recognizer for an unweighted graph (alist of node -> neighbor list)
(defun graphp (g)
  (if (endp g)
      t
    (and (consp (car g))
         (true-listp (cdar g))
         (graphp (cdr g)))))

; Get the neighbors of a node in graph g
(defun neighbors (node g)
  (cdr (assoc-equal node g)))

; Get all nodes in the graph
(defun nodes (g)
  (strip-cars g))

; Is there an edge from u to v?
(defun edge-p (u v g)
  (member-equal v (neighbors u g)))

### Example Graph

Let's define a sample undirected graph:

```
    A --- B
    |     |
    C --- D --- E
```

In [ ]:
; An undirected graph — each edge appears in both directions
(defconst *example-graph*
  '((a . (b c))
    (b . (a d))
    (c . (a d))
    (d . (b c e))
    (e . (d))))

; Test the graph operations
(list (graphp *example-graph*)
      (neighbors 'd *example-graph*)
      (nodes *example-graph*)
      (edge-p 'a 'b *example-graph*)
      (edge-p 'a 'e *example-graph*))

## 2. Graph Traversal: Depth-First Search (DFS)

DFS explores a graph by going as deep as possible along each branch before backtracking. We use the *accumulator pattern* to track visited nodes.

- Maintain a `visited` list to avoid revisiting nodes
- For each unvisited neighbor, recursively visit it
- The accumulator grows monotonically (we only add, never remove)

In [ ]:
; Depth-First Search
; Returns the list of all nodes reachable from 'start'
(defun dfs-visit (node g visited)
  (if (member-equal node visited)
      visited
    (dfs-visit-neighbors (neighbors node g)
                         g
                         (cons node visited))))

(defun dfs-visit-neighbors (nbrs g visited)
  (if (endp nbrs)
      visited
    (dfs-visit-neighbors (cdr nbrs)
                         g
                         (dfs-visit (car nbrs) g visited))))

In [ ]:
; Run DFS starting from node A
(dfs-visit 'a *example-graph* nil)

### DFS Properties

In [ ]:
; Property: visited list only grows (monotonicity)
(defthm dfs-visit-monotonic
  (subsetp-equal visited
                 (dfs-visit node g visited)))

; Property: the start node is in the result
(defthm start-in-dfs
  (member-equal node
                (dfs-visit node g visited)))

## 3. Breadth-First Search (BFS)

BFS explores a graph level by level using a queue. It visits all neighbors of the current node before going deeper.

| Property | DFS | BFS |
|----------|-----|-----|
| Data structure | Stack (call stack) | Queue |
| Order | Deep first | Broad first |
| Shortest path? | No | Yes (unweighted) |
| Space | $O(\text{depth})$ | $O(\text{width})$ |

In [ ]:
; Breadth-First Search using a queue
(defun bfs-step (queue g visited)
  (declare (xargs :measure (let ((unseen (set-difference-equal (nodes g) visited)))
                             (+ (len unseen) (len queue)))))
  (if (endp queue)
      visited
    (let ((node (car queue))
          (rest (cdr queue)))
      (if (member-equal node visited)
          (bfs-step rest g visited)
        (let ((new-nbrs (set-difference-equal
                          (neighbors node g)
                          visited)))
          (bfs-step (append rest new-nbrs)
                    g
                    (cons node visited)))))))

; BFS entry point
(defun bfs (start g)
  (bfs-step (list start) g nil))

In [ ]:
; Run BFS from node A
(bfs 'a *example-graph*)

In an unweighted graph, BFS visits nodes in order of their distance from the start. The first time BFS reaches a node, it has found the shortest path (fewest edges).

## 4. Shortest Paths: Dijkstra's Algorithm

For *weighted* graphs, we need Dijkstra's algorithm. The ACL2 community books contain a fully verified implementation in `books/misc/dijkstra-shortest-path.lisp` with 126 theorems. We present a simplified version here.

**Key Concepts:**
- **Distance table**: alist mapping each node to its current best known distance
- **Relaxation**: when we find a shorter path, update the distance
- **Priority selection**: always process the unvisited node with smallest distance

In [ ]:
; Weighted graph
(defconst *weighted-graph*
  '((a . ((b . 4) (c . 2)))
    (b . ((d . 3) (e . 1)))
    (c . ((b . 1) (d . 5)))
    (d . ((e . 2)))
    (e . ())))

; Get weighted neighbors and edge weight
(defun weighted-neighbors (node g)
  (cdr (assoc-equal node g)))

(defun edge-weight (u v g)
  (cdr (assoc-equal v (weighted-neighbors u g))))

In [ ]:
; Distance table operations ('inf for infinity)
(defun get-dist (node dist-table)
  (let ((pair (assoc-equal node dist-table)))
    (if pair (cdr pair) 'inf)))

(defun set-dist (node d dist-table)
  (acons node d dist-table))

(defun dist< (a b)
  (cond ((equal b 'inf) (not (equal a 'inf)))
        ((equal a 'inf) nil)
        (t (< a b))))

In [ ]:
; Relax a single edge and all neighbors of a node
(defun relax-edge (u v w dist-table)
  (let ((du (get-dist u dist-table))
        (dv (get-dist v dist-table)))
    (if (and (not (equal du 'inf))
             (dist< (+ du w) dv))
        (set-dist v (+ du w) dist-table)
      dist-table)))

(defun relax-neighbors (u nbrs dist-table)
  (if (endp nbrs)
      dist-table
    (let* ((v (caar nbrs))
           (w (cdar nbrs))
           (new-dist (relax-edge u v w dist-table)))
      (relax-neighbors u (cdr nbrs) new-dist))))

In [ ]:
; Find the unvisited node with smallest distance
(defun find-min-node (nodes dist-table visited)
  (if (endp nodes)
      nil
    (let ((node (car nodes)))
      (if (member-equal node visited)
          (find-min-node (cdr nodes) dist-table visited)
        (let ((rest-min (find-min-node (cdr nodes) dist-table visited)))
          (if (null rest-min)
              node
            (if (dist< (get-dist node dist-table)
                       (get-dist rest-min dist-table))
                node
              rest-min)))))))

In [ ]:
; Main loop: pick min node, relax, repeat
(defun dijkstra-step (g all-nodes dist-table visited)
  (let ((u (find-min-node all-nodes dist-table visited)))
    (if (null u)
        (mv dist-table visited)
      (let* ((nbrs (weighted-neighbors u g))
             (new-dist (relax-neighbors u nbrs dist-table))
             (new-visited (cons u visited)))
        (dijkstra-step g all-nodes new-dist new-visited)))))

; Dijkstra entry point
(defun dijkstra (source g)
  (let* ((all-nodes (strip-cars g))
         (dist-table (set-dist source 0 nil)))
    (dijkstra-step g all-nodes dist-table nil)))

In [ ]:
; Run Dijkstra from node A
(dijkstra 'a *weighted-graph*)

The full correctness proof establishes **optimality** (distances are truly shortest), the **triangle inequality** ($d(v) \leq d(u) + w$), and **termination** (the visited set grows at each step).

## 5. Path Finding

Beyond distances, we often need actual paths. We define a path predicate and a path-finding function:

In [ ]:
; A path is valid if consecutive nodes are connected by edges
(defun pathp (path g)
  (cond ((endp path) t)
        ((endp (cdr path)) (member-equal (car path) (nodes g)))
        (t (and (edge-p (car path) (cadr path) g)
                (pathp (cdr path) g)))))

In [ ]:
; Find a path from start to end using DFS
(defun find-path (start end g visited)
  (cond ((equal start end) (list start))
        ((member-equal start visited) nil)
        (t (find-path-neighbors
             (neighbors start g)
             start end g
             (cons start visited)))))

(defun find-path-neighbors (nbrs start end g visited)
  (if (endp nbrs)
      nil
    (let ((sub-path (find-path (car nbrs) end g visited)))
      (if sub-path
          (cons start sub-path)
        (find-path-neighbors (cdr nbrs) start end g visited)))))

In [ ]:
; Find a path from A to E and verify it
(let ((p (find-path 'a 'e *example-graph* nil)))
  (list p (pathp p *example-graph*)))

In [ ]:
; The path returned by find-path (if non-nil) is valid
(defthm find-path-is-valid
  (implies (find-path start end g visited)
           (pathp (find-path start end g visited) g)))

## 6. Cycle Detection

A *cycle* is a path that starts and ends at the same node. The classic DFS-based detection uses three colors:
- **White**: not yet visited
- **Gray**: currently being explored (on the DFS stack)
- **Black**: fully explored

A *back edge* (an edge to a gray node) indicates a cycle in a directed graph.

In [ ]:
; Color map operations
(defun get-color (node colors)
  (let ((pair (assoc-equal node colors)))
    (if pair (cdr pair) 'white)))

(defun set-color (node color colors)
  (acons node color colors))

In [ ]:
; DFS-based cycle detection for directed graphs
(defun detect-cycle-visit (node g colors)
  (let ((colors (set-color node 'gray colors)))
    (mv-let (cycle-found colors)
      (detect-cycle-neighbors (neighbors node g) g colors)
      (if cycle-found
          (mv t colors)
        (mv nil (set-color node 'black colors))))))

(defun detect-cycle-neighbors (nbrs g colors)
  (if (endp nbrs)
      (mv nil colors)
    (let ((v (car nbrs))
          (c (get-color (car nbrs) colors)))
      (cond
        ((equal c 'gray) (mv t colors))  ; back edge = cycle!
        ((equal c 'white)
         (mv-let (cycle-found colors)
           (detect-cycle-visit v g colors)
           (if cycle-found
               (mv t colors)
             (detect-cycle-neighbors (cdr nbrs) g colors))))
        (t (detect-cycle-neighbors (cdr nbrs) g colors))))))

In [ ]:
; Check all nodes for cycles
(defun has-cycle (g)
  (has-cycle-nodes (nodes g) g nil))

(defun has-cycle-nodes (node-list g colors)
  (if (endp node-list)
      nil
    (if (equal (get-color (car node-list) colors) 'white)
        (mv-let (cycle-found colors)
          (detect-cycle-visit (car node-list) g colors)
          (if cycle-found
              t
            (has-cycle-nodes (cdr node-list) g colors)))
      (has-cycle-nodes (cdr node-list) g colors))))

In [ ]:
; Test: cyclic vs acyclic graphs
(defconst *cyclic-graph*
  '((a . (b)) (b . (c)) (c . (a))))

(defconst *dag*
  '((a . (b c)) (b . (d)) (c . (d)) (d . ())))

(list (has-cycle *cyclic-graph*)   ; T
      (has-cycle *dag*))           ; NIL

## 7. Exercises

### Exercise 5.1: Connected Components

Define a function `(connected-components g)` that returns a list of lists, where each inner list is the set of nodes in one connected component of an undirected graph.

*Hint:* Use DFS from each unvisited node to collect one component.

In [ ]:
; Exercise 5.1: Connected components
; YOUR CODE HERE


### Exercise 5.2: Topological Sort

Define a function `(topological-sort g)` that returns a topological ordering of a DAG — every node appears before all nodes it has edges to.

*Hint:* Use DFS and add each node to the result *after* visiting all its neighbors (reverse postorder).

In [ ]:
; Exercise 5.2: Topological sort
; YOUR CODE HERE


### Exercise 5.3: Shortest Path Reconstruction

Extend Dijkstra's algorithm to compute a *predecessor table*, then define `(reconstruct-path pred-table source dest)` that returns the actual shortest path. Prove the reconstructed path is valid.

In [ ]:
; Exercise 5.3: Dijkstra with path reconstruction
; YOUR CODE HERE


### Exercise 5.4: Bipartite Check

A graph is *bipartite* if its nodes can be 2-colored such that no edge connects same-colored nodes. Define `(bipartite-p g)` using BFS with 2-coloring.

In [ ]:
; Exercise 5.4: Bipartite check
; YOUR CODE HERE


## Summary

In this module, we learned to:

- **Represent graphs** as adjacency lists (alists) in ACL2
- **Traverse graphs** with DFS (stack-based) and BFS (queue-based)
- **Find shortest paths** using Dijkstra's algorithm with distance table relaxation
- **Find paths** and prove they are valid
- **Detect cycles** using the white-gray-black DFS coloring technique

| Proof Technique | Used For |
|----------------|----------|
| Accumulator monotonicity | DFS visited list only grows |
| Measure functions | Proving BFS/DFS terminate |
| Invariant maintenance | Dijkstra's distance table properties |
| Structural induction | Properties over graph structure |

---

**Next Module:** [Module 6: Machine Models — The M1 Stack Machine](06_machine_models.ipynb) — We use ACL2 to model and verify programs running on a simple JVM-like stack machine.